## PageIndex - VectorLess RAG 

**Reasoning-Based RAG with No Vector DB ,No Chuncking**

**Builds a tree index from a PDF**

	**LLM Tree Search — reasoning over structure**

	**Full end-to-end Vectorless RAG pipeline**

	**Expert-guided retrieval (domain knowledge injection)**

	**Chat API — zero LLM setup**

**Concept**
**Traditional RAG → chunk → embed → cosine similarity → retrieve**

**PageIndex RAG → build tree → LLM reasons over tree → retrieve exact sections**

**The problem with vector RAG:**

**Similarity ≠ Relevance**

A chunk about "market conditions" may score higher than the actual answer section just because it shares more words with your query.**

In [32]:
import os,json,time
from dotenv import load_dotenv

In [33]:
load_dotenv()

PAGEINDEX_API_KEY = os.getenv("PAGE_Index")
Groq_API_KEY    = os.getenv("GROQ_API_KEY")

print("PageIndex key loaded:", "✅" if PAGEINDEX_API_KEY else "❌ Missing!")
print("OpenAI key loaded:   ", "✅" if Groq_API_KEY    else "❌ Missing!")

PageIndex key loaded: ✅
OpenAI key loaded:    ✅


In [34]:
from pageindex import PageIndexClient
from langchain_groq import ChatGroq

pi_client     = PageIndexClient(api_key=PAGEINDEX_API_KEY)
Groq_client = ChatGroq(api_key=Groq_API_KEY,
                        model="llama3-8b-8192")

print("✅ PageIndex client ready")
print("✅ OpenAI client ready")

✅ PageIndex client ready
✅ OpenAI client ready


## Uploading and Indexing A PDF

**here:**

Upload your PDF or load data to the PageIndex cloud

PageIndex uses an LLM to read the document structure

Builds a hierarchical tree index (like a smart Table of Contents)

Returns a doc_id for all future operations

Why NO chunking?
Instead of cutting the document into arbitrary 500-token pieces, PageIndex respects the document's natural section boundaries — chapters, sub-sections, paragraphs — as the author intended.

In [35]:
import os
print(os.getcwd())

d:\Documents\PROJECTS\RAG\PageIndex


In [38]:
# from pathlib import Path

# PDF_DIR = Path("../data/pdf")

# for file_path in PDF_DIR.glob("*.pdf"):
#     print(f"Uploading: {file_path.name}")

#     result = pi_client.submit_document(str(file_path))
#     print(f"✅ Uploaded → {result['doc_id']}")

In [41]:
doc_id = "pi-cmomy0tnl00ku01qr1s2q2gmr"

status_result = pi_client.get_document(doc_id)
print(status_result)

PageIndexAPIError: Failed to get document metadata: {"detail":"Document not found"}

In [39]:
# ── Poll until processing is complete ───────────────────────────────────────
# PageIndex builds the tree asynchronously.
# For a 50-page PDF this typically takes 30–90 seconds.

import time

print("⏳ Building tree index...")
print("   (This runs once per document — the index is cached for reuse)")

while True:
    status_result = pi_client.get_document(doc_id)
    status = status_result.get("status")

    print(f"   Status: {status}")

    if status == "completed":
        print("\n✅ Tree index ready!")
        break

    elif status == "failed":
        print("\n❌ Processing failed. Check PDF format or PageIndex limits/quota.")
        break

    elif status in ["processing", "queued", "indexing"]:
        # normal intermediate states
        pass

    else:
        print(f"⚠️ Unknown status: {status}")

    time.sleep(5)

⏳ Building tree index...
   (This runs once per document — the index is cached for reuse)


PageIndexAPIError: Failed to get document metadata: {"detail":"Document not found"}

## Section 3: Inspect the Tree Structure

What the tree looks like:

Document
: 1706.03762v7.pdf
├── Abstract
│   └── (summary of attention mechanism paper)
├── Introduction
│   ├── Background: sequence models
│   └── Problem statement
├── Model Architecture
│   ├── Encoder
│   ├── Decoder
│   └── Attention Mechanism
├── Experiments
│   ├── Dataset description
│   └── Results
└── Conclusion 

node_id — unique ID used during retrieval

title — section heading

page_index — page number in original PDF

text — section summary (when node_summary=True)

nodes — child sections (nested)

**This structure is what the LLM reasons over during retrieval.**

In [ ]:
# ── Fetch the full tree ─────────────────────────────────────────────────────
import json

tree_result = pi_client.get_tree(doc_id, node_summary=True)
pageindex_tree = tree_result.get("result", [])

print(f"📊 Top-level sections: {len(pageindex_tree)}")

if not pageindex_tree:
    print("\n⚠️ Tree is empty. Document may still be processing or failed.")
else:
    print("\n🌲 Top-level nodes:")
    
    # print all top-level sections (important for structured PDFs)
    for node in pageindex_tree:
        print(f"- [{node.get('node_id')}] {node.get('title')} (p.{node.get('page_index')})")

    print("\n🌲 Raw tree (first node):")
    print(json.dumps(pageindex_tree[0], indent=2))

📊 Top-level sections: 1

🌲 Top-level nodes:
- [0000] A Sample Research Proposal with Comments (p.1)

🌲 Raw tree (first node):
{
  "title": "A Sample Research Proposal with Comments",
  "node_id": "0000",
  "page_index": 1,
  "summary": "The text outlines the research proposal process for students, including deadlines and required components such as problem statement, objectives, methodology, and a time schedule. It provides a sample research proposal with detailed comments on each section, covering the introduction, problem statement, objectives, preliminary literature review, and methodology. The sample proposal focuses on developing a conceptual framework for scheduling constraint management in construction projects, highlighting the importance of identifying, classifying, modeling, and resolving constraints. It discusses the limitations of current scheduling methods and the need for a more comprehensive approach. The methodology involves literature review and conceptual modeling, wi

In [ ]:
# ── Pretty-print the full tree ─
def print_tree(nodes, indent=0):
    """Recursively print tree titles for a visual overview."""

    for node in nodes:
        prefix = "  " * indent + ("└─ " if indent > 0 else "")

        node_id = node.get("node_id", "unknown")
        title   = node.get("title", "No Title")
        page    = node.get("page_index", "?")

        print(f"{prefix}[{node_id}] {title}  (p.{page})")

        children = node.get("nodes", [])
        if children:
            print_tree(children, indent + 1)


print("📚 Full Document Structure:\n")
print_tree(pageindex_tree)

📚 Full Document Structure:

[0000] A Sample Research Proposal with Comments  (p.1)


In [ ]:
# ── Count total nodes ────────────────────────────────────────────────────────
def count_nodes(nodes):
    total = len(nodes)
    for n in nodes:
        if n.get("nodes"):
            total += count_nodes(n["nodes"])
    return total

total = count_nodes(pageindex_tree)
print(f"🔢 Total nodes in tree: {total}")
print("   Each node = one retrievable section of the document")


## Section 4: LLM Tree Search — The Core of PageIndex

This is where PageIndex fundamentally differs from vector RAG.

Vector RAG retrieval:
query → embed → cosine_similarity(query_vec, all_chunk_vecs) → top-k chunks
Problem: finds what's similar, not what's relevant

PageIndex retrieval:
query + tree → LLM reasons → "node 0007 and 0008 contain the answer"
Advantage: LLM understands document structure, context, and intent

The LLM acts like a human expert scanning a Table of Contents.

In [ ]:
# ── LLM Tree Search Function ─────────────────────────────────────────────────

def llm_tree_search(query: str, tree: list, model: str = "gpt-4o") -> dict:
    """
    Core PageIndex retrieval:
    Sends the query + document tree to an LLM.
    LLM reasons over the structure and returns relevant node_ids.
    
    Returns: dict with 'thinking' (reasoning) and 'node_list' (node IDs)
    """
    
    # Compress tree to save tokens — only send titles + short summaries
    def compress(nodes):
        out = []
        for n in nodes:
            entry = {
                "node_id": n["node_id"],
                "title":   n["title"],
                "page":    n.get("page_index", "?"),
                "summary": n.get("text", "")[:150]  # first 150 chars
            }
            if n.get("nodes"):
                entry["children"] = compress(n["nodes"])
            out.append(entry)
        return out
    
    compressed_tree = compress(tree)
    
    prompt = f"""You are given a query and a document's tree structure (like a Table of Contents).
Your task: identify which node IDs most likely contain the answer to the query.
Think step-by-step about which sections are relevant.

Query: {query}

Document Tree:
{json.dumps(compressed_tree, indent=2)}

Reply ONLY in this exact JSON format:
{{
  "thinking": "<your step-by-step reasoning>",
  "node_list": ["node_id1", "node_id2"]
}}"""

    response = openai_client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"}
    )
    
    return json.loads(response.choices[0].message.content)

In [ ]:

# ── Test with a sample query ─────────────────────────────────────────────────
query = "What is the syllabus covered in Modern LLM finetuning?"

print(f"🔍 Query: {query}\n")
result = llm_tree_search(query, pageindex_tree)

print("🧠 LLM Reasoning:")
print(result.get("thinking", "N/A"))
print()
print("🎯 Selected Node IDs:", result.get("node_list", []))